# 🚀 머신러닝 실습 : 고객 구매 데이터로 성별 예측 모델링 (분류 문제)

* 주어진 데이터는 백화점 고객의 1년 간 구매 데이터입니다.
* 고객 3,500명에 대한 학습용 데이터(y.csv, X.csv)를 이용하여 성별예측 모형을 만들어보세요.
* 모델의 성능은 자유롭게 측정해봅니다!

[실습 프로세스]
1. 데이터 불러오기  
2. 데이터 탐색
3. 데이터 전처리  
4. 학습/테스트 데이터 분리  
5. 모델 선택 및 학습  
6. 예측 및 평가  

# 0. 라이브러리 불러오기
라이브러리를 가져와서 과정을 준비합니다

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings(action='ignore')

# 1. 데이터 불러오기
데이터를 가져와서 과정을 준비합시다.
인코딩 방식은 'euc-kr' 을 활용하세요.

데이터 출처 : 한국데이터산업진흥원 빅데이터분석기사 실기 공개 예시 문항

독립 변수 데이터셋 : ./data/X.csv

종속 변수 데이터셋 : ./data/y.csv

데이터 파일을 불러옵니다. 보통 CSV 파일을 pandas로 읽어옵니다.

In [ ]:
# import os
# # 노트북 파일이 있는 폴더로 이동 (예시)
# os.chdir(r'C:\githome\hipython_rep')

# # 변경 후 확인
# print("변경 후:", os.getcwd())

변경 후: c:\githome\hipython_rep


In [2]:
X = pd.read_csv('./data/X.csv', encoding='euc-kr')
y = pd.read_csv('./data/y.csv', encoding='euc-kr')

# 실습 환경에 맞춰 경로 조정 가능
# X = pd.read_csv('X.csv', encoding='cp949')
# y = pd.read_csv('y.csv', encoding='cp949')

In [3]:
X.head()

,cust_id,총구매액,최대구매액,환불금액,주구매상품,주구매지점,내점일수,내점당구매건수,주말방문비율,구매주기
0,0,68282840,11264000,6860000.0,기타,강남점,19,3.894737,0.527027,17
1,1,2136000,2136000,300000.0,스포츠,잠실점,2,1.500000,0.000000,1
2,2,3197000,1639000,NaN,남성 캐주얼,관악점,2,2.000000,0.000000,1
3,3,16077620,4935000,NaN,기타,광주점,18,2.444444,0.318182,16
4,4,29050000,24000000,NaN,보석,본 점,2,1.500000,0.000000,85


In [4]:
y.head()

,cust_id,gender
0,0,0
1,1,0
2,2,1
3,3,1
4,4,0


# 2. 데이터 탐색하기
데이터를 이해할 수 있도록 탐색과정을 수행해봅시다.
데이터의 상위 몇 개 행을 출력하여 전체 구조를 미리 확인합니다.

데이터의 요약 정보나 통계 정보를 출력해 변수들의 유형과 분포를 확인합니다.

데이터의 요약 정보나 통계 정보를 출력해 변수들의 유형과 분포를 확인합니다.

In [ ]:
# 데이터 기본 정보 확인
print('=== X 데이터 정보 ===')
print(X.info())
print()
print('=== y 데이터 정보 ===')
print(y.info())

In [ ]:
# 기술 통계 정보 확인
print('=== X 기술통계 ===')
display(X.describe())
print()
print('=== y 타겟 분포 ===')
print(y['gender'].value_counts())
print('0: 남성, 1: 여성')

# 3. 데이터 전처리
전처리 과정을 통해서 머신러닝에 사용할 수 있는 형태의 데이터 준비
필요한 라이브러리를 불러옵니다.

인코딩 : LabelEncoder
데이터 표준화 : StandardScaler
단순히 1부터의 숫자를 부여한 'cust_id'를 수치형 변수로 받아들이면, 결과가 왜곡될 수 있으니 컬럼을 제거합니다.
데이터에 결측치가 있는지 확인해보세요
결측치에 0으로 채워 넣어 모델 학습에 지장이 없도록 합니다.
문자형 범주 데이터를 숫자로 바꾸기 위한 인코딩을 수행합니다.

각 데이터에 표준화를 적용하여 데이터의 스케일(크기 차이)을 맞춰줍니다.

평균을 0, 표준편차를 1로 맞춰서 → 데이터가 정규 분포 형태로 변환되도록 하세요

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
# cust_id 컬럼 제거 (의미 없는 식별자)
X = X.drop(columns=['cust_id'])
y = y.drop(columns=['cust_id'])

print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
# 결측치 확인
print('=== X 결측치 현황 ===')
print(X.isnull().sum())

In [ ]:
# 결측치 0으로 채우기
X = X.fillna(0)

# 결측치 처리 확인
print('처리 후 결측치:')
print(X.isnull().sum())

In [ ]:
# 문자형 범주형 컬럼 Label Encoding
le = LabelEncoder()

cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
print('인코딩 대상 컬럼:', cat_cols)

for col in cat_cols:
    X[col] = le.fit_transform(X[col])

print('인코딩 완료')
X.head()

In [ ]:
# 학습 / 테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# StandardScaler 표준화 (평균 0, 표준편차 1)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # 훈련 데이터로 fit 후 transform
X_test  = scaler.transform(X_test)        # 테스트 데이터는 transform만

print('X_train shape:', X_train.shape)
print('X_test  shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test  shape:', y_test.shape)

In [ ]:
# 공통 평가 함수 정의
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

def get_clf_eval(model_name, y_test, pred):
    cm        = confusion_matrix(y_test, pred)
    accuracy  = accuracy_score (y_test, pred)
    precision = precision_score(y_test, pred)
    recall    = recall_score   (y_test, pred)
    f1        = f1_score       (y_test, pred)
    print(f'[ {model_name} ]')
    print('Confusion Matrix:')
    print(cm)
    print(f'  정확도(Accuracy) : {accuracy:.4f}')
    print(f'  정밀도(Precision): {precision:.4f}')
    print(f'  재현율(Recall)   : {recall:.4f}')
    print(f'  F1 Score         : {f1:.4f}')
    print('-' * 40)
    return {'model': model_name, 'accuracy': accuracy,
            'precision': precision, 'recall': recall, 'f1': f1}

# 5-1. 모델링 - LogisticRegression
본격적으로 모델을 선언하고 학습시킵니다.
필요한 라이브러리를 불러옵니다.

모델을 선언하여 객체화시킵니다.

모델을 학습 데이터에 맞춰 학습시킵니다.

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr_clf = LogisticRegression(max_iter=2000, random_state=42)

In [ ]:
lr_clf.fit(X_train, y_train.values.ravel())

# 6-1. 예측 성능 확인해보기 - LogisticRegression
학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.
학습시킨 모델의 성능을 알아봅니다
각 평가지표로 모델의 성능을 수치화하여 확인합니다.
필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [ ]:
lr_pred = lr_clf.predict(X_test)
results = []
results.append(get_clf_eval('LogisticRegression', y_test, lr_pred))

# 5-2. 모델링 - DecisionTreeClassifier
본격적으로 모델을 선언하고 학습시킵니다.
필요한 라이브러리를 불러옵니다.

모델을 선언하여 객체화시킵니다.

모델을 학습 데이터에 맞춰 학습시킵니다.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
dt_clf = DecisionTreeClassifier(random_state=42)

In [ ]:
dt_clf.fit(X_train, y_train.values.ravel())

# 6-2. 예측 성능 확인해보기 - DecisionTreeClassifier
학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.
학습시킨 모델의 성능을 알아봅니다
각 평가지표로 모델의 성능을 수치화하여 확인합니다.
필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [ ]:
dt_pred = dt_clf.predict(X_test)
results.append(get_clf_eval('DecisionTreeClassifier', y_test, dt_pred))

# 5-3. 모델링 - RandomForestClassifier
본격적으로 모델을 선언하고 학습시킵니다.
필요한 라이브러리를 불러옵니다.

모델을 선언하여 객체화시킵니다.

모델을 학습 데이터에 맞춰 학습시킵니다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)

In [ ]:
rf_clf.fit(X_train, y_train.values.ravel())

# 6-3. 예측 성능 확인해보기 - RandomForestClassifier
학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.
학습시킨 모델의 성능을 알아봅니다
각 평가지표로 모델의 성능을 수치화하여 확인합니다.
필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [ ]:
rf_pred = rf_clf.predict(X_test)
results.append(get_clf_eval('RandomForestClassifier', y_test, rf_pred))

# 5-4. 모델링 - XGBoost
본격적으로 모델을 선언하고 학습시킵니다.
필요한 라이브러리를 불러옵니다.

모델을 선언하여 객체화시킵니다.

모델을 학습 데이터에 맞춰 학습시킵니다.

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb_clf = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')

In [ ]:
xgb_clf.fit(X_train, y_train.values.ravel())

# 6-4. 예측 성능 확인해보기 - XGBoost
학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.
학습시킨 모델의 성능을 알아봅니다
각 평가지표로 모델의 성능을 수치화하여 확인합니다.
필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [ ]:
xgb_pred = xgb_clf.predict(X_test)
results.append(get_clf_eval('XGBoost', y_test, xgb_pred))

# 7. 위 4가지 모델의 학습 & 예측 & 평가 결과를 확인하고 최고 성능을 내는 모델을 찾아봅시다!
어떤 모델이 가장 성능이 좋은가요 ?

In [ ]:
# 4가지 모델 성능 비교 요약
results_df = pd.DataFrame(results)
results_df = results_df.set_index('model')
results_df = results_df.sort_values('f1', ascending=False)

print('=== 모델 성능 비교 (F1 기준 내림차순) ===')
display(results_df.style.highlight_max(axis=0, color='lightgreen'))

best_model = results_df.index[0]
print(f'\n🏆 최고 성능 모델: {best_model} (F1: {results_df.loc[best_model, "f1"]:.4f})')